# 04b - Train resnet18_imagenet_covidqu

This notebook runs one ResNet18 + SimCLR experiment. It only prepares Colab and calls repository scripts with `!python`.


## 1. Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Clone Or Pull Repo


In [2]:
from pathlib import Path

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('Repo:', Path.cwd())


/content
Cloning into 'contrastive-synthesis-medcls_CVProject'...
remote: Enumerating objects: 21500, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 21500 (delta 154), reused 160 (delta 99), pack-reused 21273 (from 3)
Receiving objects: 100% (21500/21500), 619.98 MiB | 34.82 MiB/s, done.
Resolving deltas: 100% (183/183), done.
Updating files: 100% (21272/21272), done.
/content/contrastive-synthesis-medcls_CVProject
Repo: /content/contrastive-synthesis-medcls_CVProject


## 3. Install Minimal Dependencies

Colab already provides PyTorch and torchvision.


In [3]:
!pip install -q scikit-learn matplotlib pandas Pillow PyYAML

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 4. Editable Runner Variables

Use `PRETRAIN_EPOCH_OVERRIDE` to control SimCLR cost. Keep `FINETUNE_EPOCH_OVERRIDE = None` to use the config fine-tuning budget of 70 epochs for fair comparison with ResNet baselines.


In [4]:
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')
OUTPUT_ROOT = Path('/content/drive/MyDrive/medcls_cvproject/results/experiments')
SYNTHETIC_MANIFEST = Path('/content/drive/MyDrive/medcls_cvproject/data/manifests/synthetic_dcgan.csv')

RUN_PRETRAIN = True
RUN_FINETUNE = True
PRETRAIN_EPOCH_OVERRIDE = 70  # Total target pretraining epochs. Increase to 20, 30, ... to resume in chunks.
FINETUNE_EPOCH_OVERRIDE = None  # Keep None for config 70 and fair baseline comparison.

pretrain_epoch_arg = '' if PRETRAIN_EPOCH_OVERRIDE is None else f'--epochs {PRETRAIN_EPOCH_OVERRIDE}'
finetune_epoch_arg = '' if FINETUNE_EPOCH_OVERRIDE is None else f'--epochs {FINETUNE_EPOCH_OVERRIDE}'

%cd {REPO_ROOT}
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('SYNTHETIC_MANIFEST:', SYNTHETIC_MANIFEST)
print('RUN_PRETRAIN:', RUN_PRETRAIN)
print('RUN_FINETUNE:', RUN_FINETUNE)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)


/content/contrastive-synthesis-medcls_CVProject
OUTPUT_ROOT: /content/drive/MyDrive/medcls_cvproject/results/experiments
SYNTHETIC_MANIFEST: /content/drive/MyDrive/medcls_cvproject/data/manifests/synthetic_dcgan.csv
RUN_PRETRAIN: True
RUN_FINETUNE: True
pretrain_epoch_arg: --epochs 70
finetune_epoch_arg: 


## 5. Lightweight Checks


In [5]:
!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_simclr_resnet.py scripts/run_classification_resnet.py



Experiment Input Check Report
[PASS] common config
  - loaded configs/experiments/common.yaml
[PASS] fixed supervised manifests
  - train: {'COVID': 578, 'Lung_Opacity': 961, 'Viral_Pneumonia': 215, 'Normal': 1630} total=3384
  - val: {'COVID': 72, 'Lung_Opacity': 120, 'Viral_Pneumonia': 26, 'Normal': 203} total=421
  - test: {'COVID': 73, 'Lung_Opacity': 121, 'Viral_Pneumonia': 28, 'Normal': 205} total=427
[PASS] experiment config files
[PASS] resnet18_covidqu
  - planned output_dir: results/experiments/resnet18_covidqu
[PASS] resnet18_covidqu_syn
  - synthetic_dcgan: {'COVID': 1000, 'Lung_Opacity': 1000, 'Viral_Pneumonia': 1000, 'Normal': 1000} total=4000
  - planned output_dir: results/experiments/resnet18_covidqu_syn
[PASS] resnet18_imagenet
  - no contrastive pretraining data required
  - planned output_dir: results/experiments/resnet18_imagenet
[PASS] resnet18_imagenet_covidqu
  - planned output_dir: results/experiments/resnet18_imagenet_covidqu
[PASS] resnet18_imagenet_covidqu_

## 6. Experiment: resnet18_imagenet_covidqu

SimCLR pretraining from ImageNet initialization on real unlabeled COVID-QU, then supervised fine-tuning on real labeled manifests.

Run `Pretrain Only` repeatedly by increasing `PRETRAIN_EPOCH_OVERRIDE`. Run `Fine-Tune Only` only after pretraining reaches the epoch target you want to report.


### 6a. Pretrain Only: resnet18_imagenet_covidqu


In [6]:
EXP = 'resnet18_imagenet_covidqu'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_simclr_checkpoint.pth'

print('OUT:', OUT)
print('CKPT:', CKPT, 'exists=', CKPT.exists())
print('RESUME_CKPT:', RESUME_CKPT, 'exists=', RESUME_CKPT.exists())

if RUN_PRETRAIN:
    !python scripts/run_simclr_resnet.py \
      --config configs/experiments/resnet18/imagenet_covidqu.yaml \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {pretrain_epoch_arg}
else:
    print('Skipping pretrain', EXP)


OUT: /content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu
CKPT: /content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/pretrain/checkpoints/best_simclr_backbone.pth exists= True
RESUME_CKPT: /content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/pretrain/checkpoints/last_simclr_checkpoint.pth exists= True
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 171MB/s]
Resuming SimCLR from /content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/pretrain/checkpoints/last_simclr_checkpoint.pth at epoch 70
Resume checkpoint already reached epoch 70; target epochs=70. Nothing to do.
Saved SimCLR checkpoint: /content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/pretrain/checkpoints/best_simclr_backbone.pth


### 6b. Fine-Tune Only: resnet18_imagenet_covidqu


In [7]:
EXP = 'resnet18_imagenet_covidqu'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'

print('CKPT:', CKPT, 'exists=', CKPT.exists())

if RUN_FINETUNE:
    if not CKPT.exists():
        raise FileNotFoundError(f'SimCLR checkpoint not found: {CKPT}. Finish pretraining first.')
    !python scripts/run_classification_resnet.py \
      --config configs/experiments/resnet18/imagenet_covidqu.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {finetune_epoch_arg}
else:
    print('Skipping finetune', EXP)


CKPT: /content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/pretrain/checkpoints/best_simclr_backbone.pth exists= True
Missing keys after SimCLR encoder load: ['fc.weight', 'fc.bias']
Epoch 1/70 train_loss=1.4286 val_loss=1.1876 val_acc=0.5416 val_f1_macro=0.3825
Epoch 2/70 train_loss=1.0433 val_loss=0.8801 val_acc=0.6793 val_f1_macro=0.4917
Epoch 3/70 train_loss=0.8206 val_loss=0.7247 val_acc=0.7553 val_f1_macro=0.6391
Epoch 4/70 train_loss=0.6939 val_loss=0.6361 val_acc=0.7838 val_f1_macro=0.6926
Epoch 5/70 train_loss=0.5996 val_loss=0.5712 val_acc=0.8005 val_f1_macro=0.7367
Epoch 6/70 train_loss=0.5373 val_loss=0.5229 val_acc=0.8147 val_f1_macro=0.7925
Epoch 7/70 train_loss=0.4852 val_loss=0.4831 val_acc=0.8361 val_f1_macro=0.8312
Epoch 8/70 train_loss=0.4566 val_loss=0.4509 val_acc=0.8432 val_f1_macro=0.8418
Epoch 9/70 train_loss=0.4081 val_loss=0.4339 val_acc=0.8480 val_f1_macro=0.8463
Epoch 10/70 train_loss=0.3795 val_loss=0.4130 val_acc=0.8646 val

## 7. Display Result


In [8]:
import json
import pandas as pd

metrics_path = OUT / 'metrics.json'
if metrics_path.exists():
    display(pd.DataFrame([{**{'experiment_id': EXP}, **json.loads(metrics_path.read_text())}]))
else:
    print('No metrics yet:', metrics_path)

!find "{OUT}" -maxdepth 3 -type f | sort || true


,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,resnet18_imagenet_covidqu,0.915691,0.927241,0.929833,0.928367,0.915283,0.915691,0.915348,30,0.916111


/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/best_checkpoint.pth
/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/classification_report.csv
/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/config_resolved_simclr.yaml
/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/config_resolved.yaml
/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/confusion_matrix.png
/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/metrics.json
/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/pretrain/checkpoints/best_simclr_backbone.pth
/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/pretrain/checkpoints/last_simclr_checkpoint.pth
/content/drive/MyDrive/medcls_cvproject/results/experiments/resnet18_imagenet_covidqu/pre